******importing libraries******

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier


****Loding csv using pandas****

In [2]:
train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')
test_ids = test['PassengerId']

train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)
test['IsAlone'] = (test['FamilySize'] == 1).astype(int)


***Cleaning data***

In [3]:
train = train.drop(columns=['Cabin','Name','Ticket','PassengerId'],errors ='ignore')
test = test.drop(columns=['Cabin','Name','Ticket','PassengerId'],errors='ignore')

train['Age'] = train['Age'].fillna(train['Age'].median())
test['Age'] = test['Age'].fillna(test['Age'].median())

train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])
test['Embarked'] = test['Embarked'].fillna(test['Embarked'].mode()[0])

test['Fare'] = test['Fare'].fillna(test['Fare'].median())

In [4]:
train = pd.get_dummies(train, drop_first=True)
test = pd.get_dummies(test, drop_first=True)

***Splitting data***

In [5]:
X = train.drop('Survived', axis=1)
y = train['Survived']
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

***Training by different modules***

In [6]:
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

pred_lr = lr.predict(X_val)

print("Logistic Accuracy:", accuracy_score(y_val, pred_lr))

Logistic Accuracy: 0.7988826815642458


In [7]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_val)

print("Random Forest Accuracy:", accuracy_score(y_val, pred_rf))

Random Forest Accuracy: 0.8268156424581006


In [8]:
dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train, y_train)

pred_dt = dt.predict(X_val)

print("Decision Tree Accuracy:", accuracy_score(y_val, pred_dt))

Decision Tree Accuracy: 0.7821229050279329


In [9]:
rfc = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_split=5,
    random_state=42
)

rfc.fit(X_train, y_train)
print(rfc.score(X_train,y_train))

0.8595505617977528


In [10]:
final_pred = rfc.predict(test)


***To submit model***

In [11]:
test_original = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

submission = pd.DataFrame({
    'PassengerId': test_original['PassengerId'],
    'Survived': final_pred
})

submission.to_csv('submission2.csv', index=False)